# 🎙️ Matraca Studio — Dublador & Clonador de Voz com IA
### Envie seu vídeo MP4 ou áudio, clone sua própria voz e exporte o conteúdo dublado exatamente no mesmo tempo do original!

Este notebook executa um pipeline completo e profissional de localização de vídeo e áudio:
1. **Entrada Universal:** Suporte para arquivos de vídeo (`.mp4`, `.mov`, `.mkv`) ou áudio (`.wav`, `.mp3`, `.m4a`) e microfone.
2. **Reconhecimento de Fala (Whisper):** Transcreve com alta precisão e detecta automaticamente o idioma original (executado apenas uma vez para otimização máxima).
3. **Seleção Multi-Idioma (Checkboxes):** Escolha um ou vários idiomas de destino (**Inglês, Espanhol, Francês, Alemão, Chinês, Árabe**, etc.) para dublagem em lote sequencial.
4. **Tradução Automática Segmentada:** Tradução robusta sem limite de caracteres por final de frase natural.
5. **Clonagem de Voz com IA (OmniVoice):** Recria a sua voz falando em cada idioma selecionado mantendo timbre, entonação e características vocais únicas.
6. **Sincronização Temporal com o Vídeo Original:** Aplica time-stretching inteligente com preservação total de tom (*pitch-preserved atempo* via FFmpeg), garantindo que cada áudio dublado case perfeitamente com a duração do vídeo.
7. **Download Individual de Mídia:** Gera e disponibiliza para **download individual** cada áudio WAV sincronizado e vídeo MP4 dublado gerado para cada idioma selecionado.

> ⚠️ **Requisito Obrigatório (GPU T4):**
> Vá no menu superior do Colab em **Ambiente de Execução (Runtime)** ➔ **Alterar tipo de ambiente de execução (Change runtime type)** ➔ Selecione **T4 GPU**.


In [ ]:
# @title Passo 1: Instalar Dependências e FFmpeg
# @markdown Instala OmniVoice, Whisper, Gradio, Deep-Translator e ferramentas de mídia.

!apt-get -y update -qq && apt-get -y install -qq ffmpeg
!pip install -q omnivoice gradio openai-whisper deep-translator
!pip install -q torchaudio --extra-index-url https://download.pytorch.org/whl/cu128
print('✅ Dependências e ferramentas de mídia instaladas com sucesso!')

In [ ]:
# @title Passo 2: Carregar os Modelos de IA na GPU (T4 / A100 / L4)
# @markdown Baixa e carrega o Whisper e o OmniVoice na memória de vídeo da GPU.

import os
import torch
import torchaudio
import whisper
from omnivoice import OmniVoice

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
dtype  = torch.float16 if torch.cuda.is_available() else torch.float32

print(f'🖥️ Dispositivo em uso: {device}')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'🚀 GPU Detectada: {gpu_name} ({vram:.1f} GB VRAM)')
else:
    print('⚠️ GPU não detectada! Por favor, ative a T4 GPU no menu do Colab (Runtime > Change runtime type).')

print('\n⏳ Carregando modelo Whisper (base) para transcrição de áudio...')
whisper_model = whisper.load_model('base', device=device)
print('✅ Whisper pronto!')

print('\n⏳ Carregando OmniVoice da k2-fsa (primeira vez baixa os pesos do modelo)...')
omnivoice_model = OmniVoice.from_pretrained('k2-fsa/OmniVoice', device_map=device, dtype=dtype)
print('✅ OmniVoice carregado e pronto para clonar sua voz!')

In [ ]:
# @title Passo 3: Motor de Processamento, Sincronização Temporal e Remuxing de Vídeo
# @markdown Funções auxiliares para extração de áudio, medição de duração, ajuste atempo e geração do MP4 final.

import subprocess
import json
import tempfile
import os
import re
import gc
import time
import torch
import torchaudio
from deep_translator import GoogleTranslator

def get_media_info(file_path):
    """Retorna a duração em segundos e se o arquivo contém faixa de vídeo."""
    if not file_path or not os.path.exists(file_path):
        return 0.0, False
    cmd = [
        'ffprobe', '-v', 'error',
        '-show_entries', 'format=duration:stream=codec_type',
        '-of', 'json', file_path
    ]
    try:
        res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        data = json.loads(res.stdout)
        duration = float(data.get('format', {}).get('duration', 0.0))
        streams = data.get('streams', [])
        has_video = any(s.get('codec_type') == 'video' for s in streams)
        return duration, has_video
    except Exception as e:
        print(f'Erro ao inspecionar mídia com ffprobe: {e}')
        return 0.0, False

def extract_audio_to_wav(media_path, output_wav, sample_rate=24000):
    """Converte qualquer mídia (vídeo ou áudio) em WAV mono limpo para o OmniVoice."""
    cmd = [
        'ffmpeg', '-y', '-i', media_path,
        '-vn', '-acodec', 'pcm_s16le', '-ar', str(sample_rate), '-ac', '1',
        output_wav
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)

def extract_audio_slice(input_wav, start_sec, end_sec, output_slice_wav):
    """
    Extrai uma fatia de áudio (ex: 4 a 10s) para servir de referência vocal ao OmniVoice,
    aplicando filtro passa-alta leve e normalização de volume para máxima nitidez de timbre.
    """
    duration = max(1.0, end_sec - start_sec)
    cmd = [
        'ffmpeg', '-y',
        '-ss', f'{start_sec:.3f}',
        '-t', f'{duration:.3f}',
        '-i', input_wav,
        '-af', 'highpass=f=60,loudnorm=I=-16:TP=-1.5:LRA=11',
        '-acodec', 'pcm_s16le', '-ar', '24000', '-ac', '1',
        output_slice_wav
    ]
    try:
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    except Exception:
        cmd_fb = [
            'ffmpeg', '-y',
            '-ss', f'{start_sec:.3f}',
            '-t', f'{duration:.3f}',
            '-i', input_wav,
            '-acodec', 'pcm_s16le', '-ar', '24000', '-ac', '1',
            output_slice_wav
        ]
        subprocess.run(cmd_fb, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)

def translate_text_robust(text, target_code, max_chunk=2000):
    """
    Traduz textos de qualquer tamanho garantindo o idioma de destino correto.
    Suporta pt-BR, zh-CN, es, en, etc., com fallback inteligente e pausas anti-rate-limit.
    """
    if not text or not text.strip():
        return ''
    clean_text = text.strip()

    norm_code = target_code.lower().strip()
    if norm_code in ('pt-br', 'pt_br', 'pt'):
        api_target = 'pt'
    elif norm_code in ('zh-cn', 'zh_cn', 'zh'):
        api_target = 'zh-CN'
    elif '-' in norm_code:
        api_target = norm_code.split('-')[0]
    else:
        api_target = norm_code

    if len(clean_text) < max_chunk:
        try:
            res = GoogleTranslator(source='auto', target=api_target).translate(clean_text)
            if res and res.strip():
                return res.strip()
        except Exception as e:
            print(f'⚠️ Aviso na tradução direta para {api_target}: {e}, tentando em partes...')

    sentences = re.split(r'(?<=[.!?;:\n])\s+', clean_text)
    chunks = []
    current = []
    curr_len = 0
    for s in sentences:
        s = s.strip()
        if not s:
            continue
        if curr_len + len(s) + 1 > max_chunk:
            if current:
                chunks.append(' '.join(current))
            current = [s]
            curr_len = len(s)
        else:
            current.append(s)
            curr_len += len(s) + 1
    if current:
        chunks.append(' '.join(current))

    translator = GoogleTranslator(source='auto', target=api_target)
    translated_parts = []
    for chunk in chunks:
        chunk = chunk.strip()
        if not chunk:
            continue
        part = None
        try:
            part = translator.translate(chunk)
            time.sleep(0.12)
        except Exception:
            try:
                from deep_translator import MyMemoryTranslator
                part = MyMemoryTranslator(source='auto', target=api_target).translate(chunk)
            except Exception:
                part = chunk
        if part:
            translated_parts.append(part.strip())

    result = ' '.join(translated_parts).strip()
    return result if result else clean_text

def build_atempo_filter(speed_factor):
    """Gera encadeamento de filtros atempo no FFmpeg (cada filtro suporta entre 0.5 e 2.0)."""
    speed = speed_factor
    filters = []
    while speed > 2.0:
        filters.append('atempo=2.0')
        speed /= 2.0
    while speed < 0.5:
        filters.append('atempo=0.5')
        speed /= 0.5
    filters.append(f'atempo={speed:.5f}')
    return ','.join(filters)

def time_sync_audio(synth_wav_path, target_duration, output_synced_wav, sample_rate=48000):
    """
    Ajusta a velocidade do áudio sintetizado para casar exatamente com a duração do original,
    preservando o tom e timbre da voz clonada (pitch-preserved time stretch).
    Exporta em 48.000 Hz estéreo/mono com alta qualidade para padrão de publicação no YouTube.
    """
    synth_duration, _ = get_media_info(synth_wav_path)
    if synth_duration <= 0 or target_duration <= 0:
        cmd = [
            'ffmpeg', '-y', '-i', synth_wav_path,
            '-ar', str(sample_rate),
            output_synced_wav
        ]
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        return 1.0, synth_duration

    speed_factor = synth_duration / target_duration

    if 0.985 <= speed_factor <= 1.015:
        filter_chain = f'apad=whole_dur={target_duration:.4f}'
    else:
        tempo_filter = build_atempo_filter(speed_factor)
        filter_chain = f'{tempo_filter},apad=whole_dur={target_duration:.4f}'

    cmd = [
        'ffmpeg', '-y', '-i', synth_wav_path,
        '-filter:a', filter_chain,
        '-t', f'{target_duration:.4f}',
        '-acodec', 'pcm_s16le', '-ar', str(sample_rate), '-ac', '1',
        output_synced_wav
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    return speed_factor, synth_duration

def remux_video_with_audio(original_video_path, new_audio_path, output_video_path):
    """
    Substitui a faixa de áudio do vídeo original pelo áudio dublado e sincronizado.
    Exporta em AAC 48 kHz / 256 kbps (padrão de estúdio para YouTube) sem recodificar o vídeo.
    """
    cmd = [
        'ffmpeg', '-y',
        '-i', original_video_path,
        '-i', new_audio_path,
        '-map', '0:v:0',
        '-map', '1:a:0',
        '-c:v', 'copy',
        '-c:a', 'aac',
        '-b:a', '256k',
        '-ar', '48000',
        '-shortest',
        output_video_path
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)

def generate_voice_cloning_chunked(
    model, 
    text, 
    ref_audio, 
    ref_text, 
    language='en', 
    num_steps=32, 
    speed=1.0, 
    max_chunk_chars=280,
    progress_callback=None
):
    """
    Gera fala com OmniVoice dividindo textos longos em frases naturais.
    - language: Código ISO explícito para garantir que o idioma correto seja gerado.
    - progress_callback: Atualiza o progresso granular a cada bloco para não congelar o Gradio.
    - Transições suaves e normalização de volume para máxima fidelidade e clareza acústica.
    """
    sentences = re.split(r'(?<=[.!?;:\n])\s+', text.strip())
    chunks = []
    current = []
    curr_len = 0
    for s in sentences:
        s = s.strip()
        if not s:
            continue
        if curr_len + len(s) + 1 > max_chunk_chars:
            if current:
                chunks.append(' '.join(current))
            current = [s]
            curr_len = len(s)
        else:
            current.append(s)
            curr_len += len(s) + 1
    if current:
        chunks.append(' '.join(current))

    if not chunks:
        chunks = [text.strip()] if text.strip() else ['...']

    total_chunks = len(chunks)
    print(f'🎙️ Sintetizando fala no idioma [{language}] em {total_chunks} bloco(s) natural(is)...')
    
    tensors = []
    silence = torch.zeros((1, int(24000 * 0.10)))

    for idx, c in enumerate(chunks):
        short_preview = c[:40] + ('...' if len(c) > 40 else '')
        print(f'  - [{idx+1}/{total_chunks}] ({language}) Sintetizando: {short_preview}')
        if progress_callback:
            progress_callback(idx, total_chunks, c)

        try:
            out = model.generate(
                text=c,
                language=language,
                ref_audio=ref_audio,
                ref_text=ref_text,
                num_step=int(num_steps),
                speed=float(speed)
            )
        except TypeError:
            out = model.generate(
                text=c,
                ref_audio=ref_audio,
                ref_text=ref_text,
                num_step=int(num_steps),
                speed=float(speed)
            )

        t = out[0] if isinstance(out[0], torch.Tensor) else torch.tensor(out[0])
        if t.dim() == 1:
            t = t.unsqueeze(0)
            
        tensors.append(t.cpu())
        tensors.append(silence)

    if tensors:
        tensors.pop()
        concatenated = torch.cat(tensors, dim=-1)
        max_val = torch.max(torch.abs(concatenated))
        if max_val > 0.01:
            concatenated = (concatenated / max_val) * 0.95
        return concatenated

    return torch.zeros((1, 24000))

print('✅ Motor de processamento, tradução multi-idioma e sincronização carregado!')


In [ ]:
# @title Passo 4: Iniciar a Interface de Dublagem Sincronizada (Gradio)
# @markdown Clique no botão 'Play' e acesse o link público 'Running on public URL: https://...gradio.live'

import os
import re
import shutil
import tempfile
import gc
import torch
import torchaudio
import gradio as gr

# Mapeamento estrito de idiomas suportados com códigos de tradução e IA
LANGUAGES = {
    '🇧🇷 Português Brasileiro (pt-BR)': {'code': 'pt-BR', 'deep': 'pt',    'omni': 'pt'},
    '🇺🇸 Inglês (English)':             {'code': 'en',    'deep': 'en',    'omni': 'en'},
    '🇪🇸 Espanhol (Español)':            {'code': 'es',    'deep': 'es',    'omni': 'es'},
    '🇫🇷 Francês (Français)':           {'code': 'fr',    'deep': 'fr',    'omni': 'fr'},
    '🇩🇪 Alemão (Deutsch)':             {'code': 'de',    'deep': 'de',    'omni': 'de'},
    '🇨🇳 Chinês Simplificado (中文)':    {'code': 'zh-CN', 'deep': 'zh-CN', 'omni': 'zh'},
    '🇸🇦 Árabe (العربية)':              {'code': 'ar',    'deep': 'ar',    'omni': 'ar'},
    '🇮🇹 Italiano (Italiano)':          {'code': 'it',    'deep': 'it',    'omni': 'it'},
    '🇯🇵 Japonês (日本語)':             {'code': 'ja',    'deep': 'ja',    'omni': 'ja'},
    '🇷🇺 Russo (Русский)':              {'code': 'ru',    'deep': 'ru',    'omni': 'ru'}
}

def resolve_lang_info(lang_label):
    """Garante a correspondência perfeita entre a escolha visual e os códigos internos."""
    if lang_label in LANGUAGES:
        return LANGUAGES[lang_label]
    l_lower = lang_label.lower()
    for name, info in LANGUAGES.items():
        if info['code'].lower() in l_lower or info['deep'] in l_lower or info['omni'] in l_lower:
            return info
        if 'espanh' in l_lower and 'es' in info['code'].lower():
            return info
        if 'ingl' in l_lower and 'en' in info['code'].lower():
            return info
        if 'portug' in l_lower and 'pt' in info['code'].lower():
            return info
    return {'code': 'en', 'deep': 'en', 'omni': 'en'}

def transcribe_only(media_file, progress=gr.Progress()):
    """Transcreve o áudio original com Whisper e preenche a caixa editável na interface."""
    if not media_file:
        return '', '❌ Por favor, envie um arquivo de vídeo ou áudio antes de transcrever.'
    try:
        if progress is not None:
            progress(0.15, desc='Extraindo faixa de áudio...')

        orig_dur, has_vid = get_media_info(media_file)
        temp_wav = tempfile.NamedTemporaryFile(suffix='_transcribe.wav', delete=False).name
        extract_audio_to_wav(media_file, temp_wav)

        if progress is not None:
            progress(0.50, desc='Transcrevendo fala com Whisper...')
        asr_res = whisper_model.transcribe(temp_wav)
        text = asr_res.get('text', '').strip()
        lang = asr_res.get('language', 'desconhecido')

        if progress is not None:
            progress(1.0, desc='Transcrição concluída!')

        status_msg = f"""✅ **Transcrição concluída com sucesso!**
- ⏱️ **Duração da Mídia:** `{orig_dur:.2f}s` | 🌐 **Idioma detectado:** `{lang.upper()}`
> 💡 *Você pode ler e editar qualquer palavra no campo abaixo. Em seguida, selecione os idiomas e clique em **'2. Dublar e Sincronizar'**.*"""
        return text, status_msg
    except Exception as e:
        return '', f'❌ **Erro ao transcrever áudio:** `{str(e)}`'

def process_dubbing(media_file, edited_transcription, target_lang_labels, sync_duration_opt, num_steps, user_speed, progress=gr.Progress()):
    """
    Executa a dublagem sincronizada nos idiomas escolhidos, garantindo:
    - Uso da transcrição editada pelo usuário.
    - Tradução e sintetização estrita de cada idioma escolhido (evita repetição do 1º idioma).
    - Chamada obrigatória a torch.cuda.empty_cache() e gc.collect() após cada idioma.
    - Barra de progresso contínua e sem travamento.
    """
    if not media_file:
        return (
            [], None, None,
            '❌ **Erro:** Por favor, envie um arquivo de vídeo (.mp4, .mov, etc.) ou de áudio (.wav, .mp3, etc.).',
            '', ''
        )

    if not target_lang_labels or len(target_lang_labels) == 0:
        return (
            [], None, None,
            '❌ **Erro:** Por favor, marque pelo menos um idioma de destino nas caixas de seleção.',
            '', ''
        )

    try:
        if progress is not None:
            progress(0.02, desc='Analisando arquivo original...')

        orig_duration, has_video = get_media_info(media_file)

        # Extrair áudio completo para WAV
        temp_full_wav = tempfile.NamedTemporaryFile(suffix='_full.wav', delete=False).name
        extract_audio_to_wav(media_file, temp_full_wav)

        # Transcrição: se já foi editada pelo usuário, respeita o texto editado! Caso contrário, transcreve agora.
        original_text = (edited_transcription or '').strip()
        detected_lang = 'pt'
        ref_slice_text = ""

        if not original_text:
            if progress is not None:
                progress(0.08, desc='Transcrevendo áudio original com Whisper...')
            print('🎙️ Transcrevendo áudio original com Whisper...')
            asr_result = whisper_model.transcribe(temp_full_wav)
            original_text = asr_result.get('text', '').strip()
            detected_lang = asr_result.get('language', 'desconhecido')
            segments = asr_result.get('segments', [])
            if segments:
                ref_slice_text = segments[0].get('text', '').strip()
        else:
            if progress is not None:
                progress(0.08, desc='Utilizando transcrição revisada...')
            ref_slice_text = original_text[:100]

        if not original_text:
            return (
                [], None, None,
                '❌ **Erro:** Não foi possível encontrar texto de fala para dublar. Transcreva ou digite o texto primeiro.',
                '', ''
            )

        # Amostra vocal ideal de referência tratada
        ref_slice_wav = tempfile.NamedTemporaryFile(suffix='_ref_slice.wav', delete=False).name
        ref_slice_start = 0.0
        ref_slice_end = min(orig_duration, 8.0) if orig_duration > 3.0 else orig_duration
        extract_audio_slice(temp_full_wav, ref_slice_start, ref_slice_end, ref_slice_wav)

        output_dir = tempfile.mkdtemp(prefix='matraca_batch_')
        orig_base_name = os.path.splitext(os.path.basename(media_file))[0]
        clean_base_name = re.sub(r'[^a-zA-Z0-9_\-]', '_', orig_base_name)

        generated_files = []
        translations_summary = []
        results_info = []

        total_langs = len(target_lang_labels)
        last_audio_path = None
        last_video_path = None

        print(f'🚀 Iniciando lote de dublagem para {total_langs} idioma(s): {target_lang_labels}')

        # Processamento sequencial de cada idioma selecionado
        for idx, lang_label in enumerate(target_lang_labels):
            lang_info = resolve_lang_info(lang_label)
            lang_code = lang_info['code']
            deep_code = lang_info['deep']
            omni_code = lang_info['omni']
            clean_code = lang_code.replace('-', '_')

            lang_start = 0.12 + (0.85 * idx / total_langs)
            lang_span = 0.85 / total_langs

            print(f'\n======================================================')
            print(f'🌐 [{idx+1}/{total_langs}] Idioma: {lang_label} (Deep: {deep_code} | OmniVoice: {omni_code})')
            print(f'======================================================')

            # 1. Tradução
            if progress is not None:
                progress(lang_start, desc=f'[{idx+1}/{total_langs}] Traduzindo para {lang_label}...')

            print(f'🌍 Traduzindo texto para {lang_label}...')
            translated_text = translate_text_robust(original_text, deep_code)
            translations_summary.append(f'### {lang_label} ({lang_code})\n{translated_text}\n')

            # 2. Síntese vocal clonada com OmniVoice
            cloning_start = lang_start + (lang_span * 0.15)
            cloning_span = lang_span * 0.60

            def chunk_progress(chunk_idx, num_chunks, chunk_txt):
                if progress is not None:
                    p = cloning_start + (cloning_span * (chunk_idx + 1) / max(1, num_chunks))
                    progress(p, desc=f'[{idx+1}/{total_langs}] {lang_label}: Bloco {chunk_idx+1}/{num_chunks}...')

            if progress is not None:
                progress(cloning_start, desc=f'[{idx+1}/{total_langs}] Clonando voz em {lang_label}...')

            print(f'🧬 Sintetizando fala com OmniVoice (language="{omni_code}")...')
            audio_tensor = generate_voice_cloning_chunked(
                model=omnivoice_model,
                text=translated_text,
                ref_audio=ref_slice_wav,
                ref_text=ref_slice_text,
                language=omni_code,
                num_steps=int(num_steps),
                speed=float(user_speed),
                progress_callback=chunk_progress
            )

            temp_synth_wav = tempfile.NamedTemporaryFile(suffix='_synth.wav', delete=False).name
            torchaudio.save(temp_synth_wav, audio_tensor, 24000)

            # 3. Sincronização temporal milimétrica e exportação a 48 kHz
            if progress is not None:
                progress(lang_start + (lang_span * 0.78), desc=f'[{idx+1}/{total_langs}] Sincronizando tempo do áudio...')

            final_audio_path = os.path.join(output_dir, f'{clean_base_name}_{clean_code}_audio.wav')
            speed_factor = 1.0
            synth_duration = audio_tensor.shape[-1] / 24000.0

            if sync_duration_opt and orig_duration > 0:
                print(f'⏱️ Sincronizando duração: {synth_duration:.2f}s ➔ {orig_duration:.2f}s...')
                speed_factor, synth_duration = time_sync_audio(temp_synth_wav, orig_duration, final_audio_path, sample_rate=48000)
                final_duration = orig_duration
            else:
                time_sync_audio(temp_synth_wav, synth_duration, final_audio_path, sample_rate=48000)
                final_duration = synth_duration

            last_audio_path = final_audio_path
            generated_files.append(final_audio_path)

            # 4. Remuxing do vídeo MP4 (se o arquivo original tiver vídeo)
            final_video_path = None
            if has_video:
                if progress is not None:
                    progress(lang_start + (lang_span * 0.90), desc=f'[{idx+1}/{total_langs}] Renderizando vídeo MP4 final...')
                print(f'🎬 Integrando áudio dublado ao vídeo original ({lang_label})...')
                final_video_path = os.path.join(output_dir, f'{clean_base_name}_{clean_code}_dublado.mp4')
                remux_video_with_audio(media_file, final_audio_path, final_video_path)
                last_video_path = final_video_path
                generated_files.append(final_video_path)

            results_info.append({
                'label': lang_label,
                'code': lang_code,
                'synth_dur': synth_duration,
                'final_dur': final_duration,
                'speed_factor': speed_factor,
                'audio_file': os.path.basename(final_audio_path),
                'video_file': os.path.basename(final_video_path) if final_video_path else None
            })

            # 5. LIBERAÇÃO DA MEMÓRIA GPU
            print(f'🧹 Liberando memória GPU após [{lang_label}]...')
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()

        if progress is not None:
            progress(1.0, desc='Dublagem concluída com sucesso!')

        # Tabela resumo em Markdown
        rows_md = []
        for r in results_info:
            vid_col = f"`{r['video_file']}`" if r['video_file'] else 'N/A (Entrada de Áudio)'
            rows_md.append(f"| {r['label']} | `{r['audio_file']}` | {vid_col} | `{r['speed_factor']:.3f}x` | `{r['final_dur']:.2f}s` |")

        table_body = '\n'.join(rows_md)
        status_md = f"""### ✅ Dublagem Concluída com Sucesso! ({len(results_info)} idioma(s) gerado(s))
- ⏱️ **Duração do Original:** `{orig_duration:.2f}s`
- 📥 **Downloads Disponíveis:** {len(generated_files)} arquivo(s) prontos para download individual abaixo.
- 🎧 **Qualidade Acústica:** Áudio masterizado em **48.000 Hz** com ajuste de tempo e timbre preservados.

| Idioma | Áudio WAV (48kHz) | Vídeo MP4 Dublado | Ajuste de Ritmo | Duração Final |
| :--- | :--- | :--- | :--- | :--- |
{table_body}
"""

        all_translations = '\n\n---\n\n'.join(translations_summary)
        return generated_files, last_video_path, last_audio_path, status_md, original_text, all_translations

    except Exception as e:
        import traceback
        traceback.print_exc()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
        return (
            [], None, None,
            f'❌ **Erro durante a execução:** `{str(e)}`',
            edited_transcription or '', ''
        )

def process_free_tts(custom_text, ref_audio, num_steps, speed):
    """Síntese vocal avulsa com liberação de memória."""
    if not custom_text.strip():
        return None, '❌ Digite um texto para sintetizar.'
    if not ref_audio:
        return None, '❌ Envie uma amostra de áudio com a voz a ser clonada.'
    try:
        audio_tensor = generate_voice_cloning_chunked(
            model=omnivoice_model,
            text=custom_text,
            ref_audio=ref_audio,
            ref_text='',
            language='en',
            num_steps=int(num_steps),
            speed=float(speed)
        )
        tmp_file = tempfile.NamedTemporaryFile(suffix='.wav', delete=False)
        torchaudio.save(tmp_file.name, audio_tensor, 24000)
        out_48k = tempfile.NamedTemporaryFile(suffix='_48k.wav', delete=False).name
        time_sync_audio(tmp_file.name, audio_tensor.shape[-1] / 24000.0, out_48k, sample_rate=48000)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
        return out_48k, '✅ Fala sintetizada com sucesso com a sua voz!'
    except Exception as e:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
        return None, f'❌ Erro: {str(e)}'

# Interface Gráfica Moderna
with gr.Blocks(title='Matraca Studio — Dublador & Clonador de Voz com IA', theme=gr.themes.Soft(primary_hue='emerald', secondary_hue='blue')) as demo:
    gr.HTML("""
    <div style='text-align: center; margin-bottom: 18px;'>
        <h1 style='font-size: clamp(1.2rem, 3.2vw, 2.0rem); font-weight: 700; white-space: nowrap; margin-bottom: 6px;'>🎙️ Matraca Studio — Dublador & Clonador de Voz com IA</h1>
        <p style='color: #555; font-size: 1.1em;'>
            Clone sua voz e duble qualquer vídeo MP4 ou áudio para <b>múltiplos idiomas</b>
            mantendo <b>exatamente o mesmo tempo de duração</b> do vídeo original!
        </p>
    </div>
    """)

    with gr.Tabs():
        # --- ABA 1: DUBLAGEM SINCRONIZADA ---
        with gr.TabItem('🎬 Dublagem Sincronizada (Vídeo ou Áudio)'):
            with gr.Row():
                with gr.Column(scale=1):
                    input_media = gr.File(
                        label='1. Envie seu Vídeo (MP4, MOV, MKV) ou Áudio (WAV, MP3, M4A)',
                        file_count='single',
                        type='filepath'
                    )
                    btn_transcribe = gr.Button('📝 1. Transcrever e Analisar Áudio Original', variant='secondary', size='md')
                    status_transcribe = gr.Markdown('')

                    txt_orig = gr.Textbox(
                        label='Transcrição do Áudio Original (Revise e edite as palavras se desejar):',
                        placeholder='Clique em "1. Transcrever e Analisar Áudio Original" acima para transcrever, ou digite/cole o texto aqui...',
                        lines=4,
                        interactive=True
                    )

                    target_langs = gr.CheckboxGroup(
                        choices=list(LANGUAGES.keys()),
                        value=['🇺🇸 Inglês (English)', '🇪🇸 Espanhol (Español)'],
                        label='2. Idiomas de Destino da Dublagem (Selecione um ou vários)',
                        info='Marque os idiomas desejados. Cada um será processado sequencialmente de forma otimizada.'
                    )
                    with gr.Row():
                        btn_select_all = gr.Button('☑️ Selecionar Todos', size='sm')
                        btn_clear_all = gr.Button('⬜ Limpar Seleção', size='sm')

                    sync_checkbox = gr.Checkbox(
                        value=True,
                        label='⏱️ Sincronizar Duração com o Original (Garante tempo exato para YouTube)',
                        info='Ajusta a velocidade da fala preservando o tom natural e timbre da sua voz clonada.'
                    )
                    with gr.Accordion('⚙️ Configurações Avançadas de IA', open=False):
                        steps_slider = gr.Slider(minimum=16, maximum=64, value=32, step=8, label='Passos de Difusão (Diffusion Steps - 32 para fidelidade máxima)')
                        speed_slider = gr.Slider(minimum=0.7, maximum=1.4, value=1.0, step=0.05, label='Velocidade Base da Fala')

                    btn_dub = gr.Button('✨ 2. Dublar e Sincronizar Vídeo/Áudio', variant='primary', size='lg')
                    status_label = gr.Markdown('')

                with gr.Column(scale=1):
                    output_files = gr.File(
                        label='📥 Download Individual dos Arquivos Gerados (Áudios WAV 48kHz e Vídeos MP4)',
                        file_count='multiple',
                        type='filepath',
                        interactive=False
                    )
                    with gr.Accordion('▶️ Prévia do Último Idioma Processado', open=True):
                        output_video = gr.Video(label='🎬 Vídeo Dublado Final (MP4 Sincronizado)', interactive=False)
                        output_audio = gr.Audio(label='🔊 Áudio Dublado Sincronizado (WAV 48kHz)', type='filepath', interactive=False)
                    with gr.Accordion('📜 Traduções Geradas por Idioma', open=True):
                        txt_trans = gr.Textbox(label='Traduções Geradas por Idioma', lines=6, interactive=False)

            btn_select_all.click(fn=lambda: list(LANGUAGES.keys()), outputs=[target_langs])
            btn_clear_all.click(fn=lambda: [], outputs=[target_langs])

            btn_transcribe.click(
                fn=transcribe_only,
                inputs=[input_media],
                outputs=[txt_orig, status_transcribe]
            )

            btn_dub.click(
                fn=process_dubbing,
                inputs=[input_media, txt_orig, target_langs, sync_checkbox, steps_slider, speed_slider],
                outputs=[output_files, output_video, output_audio, status_label, txt_orig, txt_trans]
            )

        # --- ABA 2: CLONAGEM LIVRE ---
        with gr.TabItem('✍️ Clonagem Livre (Digitar Texto Personalizado)'):
            gr.Markdown('Envie uma amostra de áudio com a sua voz e digite qualquer texto em qualquer idioma para sintetizar diretamente:')
            with gr.Row():
                with gr.Column(scale=1):
                    ref_audio_free = gr.Audio(label='Áudio de Referência (sua voz)', type='filepath')
                    custom_text = gr.Textbox(
                        label='Texto a ser falado',
                        placeholder='Ex: Hello everyone! Today we are introducing our new AI-powered dubbing technology.',
                        lines=4
                    )
                    with gr.Row():
                        steps_free = gr.Slider(minimum=16, maximum=64, value=32, step=8, label='Passos de Difusão')
                        speed_free = gr.Slider(minimum=0.5, maximum=1.5, value=1.0, step=0.1, label='Velocidade')
                    btn_free = gr.Button('Gerar Áudio com Minha Voz', variant='primary', size='lg')
                    status_free = gr.Markdown('')
                with gr.Column(scale=1):
                    audio_free_out = gr.Audio(label='Áudio Sintetizado (48kHz)', type='filepath', interactive=False)

            btn_free.click(
                fn=process_free_tts,
                inputs=[custom_text, ref_audio_free, steps_free, speed_free],
                outputs=[audio_free_out, status_free]
            )

demo.launch(share=True, debug=True)
